# Phase 4: Machine Learning Model Development & Empirical Benchmarking

**Project:** AI-Based Predictive Maintenance System for CNC Machines in Automotive Manufacturing Using Industrial Sensor Data  
**Objective:** Train and empirically evaluate Decision Tree, Random Forest, XGBoost, and LightGBM baseline classifiers on the SMOTE-balanced training set and untouched test set.

---

## 1. Setup, Environment, and Data Ingestion

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from config import (
    PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR, METRICS_DIR,
    RANDOM_STATE, TARGET_COL
)
from src.train_models import (
    load_processed_data,
    get_models,
    evaluate_model,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_performance_comparison,
    train_and_evaluate_all
)

# Load processed SMOTE training data and untouched test data
X_train, y_train, X_test, y_test, feature_names = load_processed_data()
print(f"SMOTE Training Set: {X_train.shape[0]:,} observations, {X_train.shape[1]} features")
print(f"Untouched Test Set: {X_test.shape[0]:,} observations, {X_test.shape[1]} features")
print(f"Test Set Class Counts: {y_test.value_counts().to_dict()} (Failures: {y_test.sum()})")

## 2. Train and Evaluate Baseline Classifiers

Trains:
1. `DecisionTreeClassifier` (Interpretable baseline)
2. `RandomForestClassifier` (Bagging ensemble)
3. `XGBClassifier` (Gradient boosted trees)
4. `LGBMClassifier` (Leaf-wise gradient boosted trees)

In [ ]:
results = train_and_evaluate_all()
perf_df = results["performance_df"]
display(perf_df)

## 3. Empirical Performance Comparison & Analysis

In [ ]:
perf_df_sorted = perf_df.sort_values("Recall", ascending=False)
print("Models Ranked by Failure Recall (Minimizing False Negatives):")
display(perf_df_sorted[["Model", "Recall", "False Negative", "Precision", "F1-Score", "PR-AUC", "Accuracy"]])

## 4. LightGBM Feature Importance

Inspects the relative split-gain importance of sensor operating features in the champion LightGBM model.

In [ ]:
lgb_model = joblib.load(MODELS_DIR / "lightgbm.joblib")
fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance (Splits)": lgb_model.feature_importances_
}).sort_values("Importance (Splits)", ascending=False)
display(fi_df)